In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

path = "/content/drive/MyDrive/Emotion-Recognition"

print(os.listdir(path))

['notebooks', 'Datasets', 'models']


In [ ]:
dataset_path = "/content/drive/MyDrive/Emotion-Recognition/Datasets"

import os
print(os.listdir(dataset_path))

['archive (3).zip']


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Emotion-Recognition/Datasets/archive (3).zip"

extract_path = "/content/drive/MyDrive/Emotion-Recognition/Datasets/RAVDESS"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted Successfully!")

Dataset Extracted Successfully!


In [ ]:
#verify Extraction
ravdess_path = "/content/drive/MyDrive/Emotion-Recognition/Datasets/RAVDESS"

print(os.listdir(ravdess_path)[:5])

['Actor_01', 'Actor_02', 'Actor_03', 'Actor_04', 'Actor_05']


In [ ]:
!pip install librosa soundfile numpy pandas scikit-learn tensorflow

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [ ]:
emotion_dict = {
    '01': 'Neutral',
    '02': 'Calm',
    '03': 'Happy',
    '04': 'Sad',
    '05': 'Angry',
    '06': 'Fearful',
    '07': 'Disgust',
    '08': 'Surprised'
}

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Emotion-Recognition/Datasets/RAVDESS"

file_paths = []
emotions = []

for actor_folder in os.listdir(dataset_path):

    actor_path = os.path.join(dataset_path, actor_folder)

    if os.path.isdir(actor_path):

        for file in os.listdir(actor_path):

            if file.endswith(".wav"):

                file_path = os.path.join(actor_path, file)

                emotion_code = file.split("-")[2]

                emotion = emotion_dict[emotion_code]

                file_paths.append(file_path)
                emotions.append(emotion)

print("Total Audio Files:", len(file_paths))
print("Sample Emotion Labels:", emotions[:10])

Total Audio Files: 1440
Sample Emotion Labels: ['Neutral', 'Neutral', 'Neutral', 'Neutral', 'Calm', 'Calm', 'Calm', 'Calm', 'Calm', 'Calm']


In [ ]:
#Create MFCC Feature Extraction Function
def extract_mfcc(file_path, n_mfcc=40):

    audio, sample_rate = librosa.load(file_path, sr=None)

    mfccs = librosa.feature.mfcc(
        y=audio,
        sr=sample_rate,
        n_mfcc=n_mfcc
    )

    mfccs_scaled = np.mean(mfccs.T, axis=0)

    return mfccs_scaled

In [ ]:
sample_features = extract_mfcc(file_paths[0])

print("MFCC Feature Shape:", sample_features.shape)
print(sample_features)

MFCC Feature Shape: (40,)
[-7.26217224e+02  6.85414200e+01  3.29339767e+00  1.22052994e+01
  5.51027870e+00  1.36674080e+01 -2.98382807e+00  3.09802866e+00
 -3.31081343e+00 -1.56438482e+00 -7.86165237e+00 -2.12428260e+00
  2.84920311e+00 -2.66780663e+00  9.59020197e-01  1.62816584e+00
 -2.73668814e+00  2.54240870e-01  2.67537785e+00 -1.76116586e+00
 -1.88647437e+00 -9.75619912e-01 -3.79437059e-01  4.00272995e-01
 -3.04404426e+00 -2.90125823e+00 -1.09248176e-01 -8.63565564e-01
 -3.33326936e+00 -1.97846520e+00  4.57081199e-01 -1.39910972e+00
 -2.92685509e+00  1.39566241e-02 -4.90734071e-01 -5.70905626e-01
  4.03992683e-02 -1.20721745e+00 -1.59498167e+00 -1.43648756e+00]


In [ ]:
X = []
y = []

for file_path, emotion in zip(file_paths, emotions):

    mfcc_features = extract_mfcc(file_path)

    X.append(mfcc_features)
    y.append(emotion)

print("Feature Extraction Completed!")

Feature Extraction Completed!


In [ ]:
X = np.array(X)
y = np.array(y)

print("Feature Shape:", X.shape)
print("Labels Shape:", y.shape)

Feature Shape: (1440, 40)
Labels Shape: (1440,)


In [ ]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print(y_encoded[:10])

[5 5 5 5 1 1 1 1 1 1]


In [ ]:
y_categorical = to_categorical(y_encoded)

print(y_categorical.shape)

(1440, 8)


In [ ]:
#Reshape data for LSTM
X = np.expand_dims(X, axis=2)

print("New Feature Shape:", X.shape)

New Feature Shape: (1440, 40, 1)


In [ ]:
#Split Training and Testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (1152, 40, 1)
Testing Shape: (288, 40, 1)


In [ ]:
#LSTM MODEL
model = Sequential()

model.add(LSTM(128, input_shape=(40,1), return_sequences=False))

model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))

model.add(Dropout(0.3))

model.add(Dense(8, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,336 (294.28 KB)

 Trainable params: 75,336 (294.28 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#MODEL TRAINING
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.1606 - loss: 2.0618 - val_accuracy: 0.1736 - val_loss: 2.0198
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2075 - loss: 1.9883 - val_accuracy: 0.2743 - val_loss: 1.9184
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2578 - loss: 1.9226 - val_accuracy: 0.3021 - val_loss: 1.8911
Epoch 4/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2378 - loss: 1.9171 - val_accuracy: 0.2014 - val_loss: 1.9515
Epoch 5/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2535 - loss: 1.8927 - val_accuracy: 0.2951 - val_loss: 1.8691
Epoch 6/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2665 - loss: 1.8793 - val_accuracy: 0.2222 - val_loss: 1.9243
Epoch 7/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2717 - loss: 1.8541 - val_accuracy: 0.3056 - val_loss: 1.8329
Epoch 8/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2786 - loss: 1.8384 - val_accuracy: 0.3090 - val_lo

In [ ]:
model.save('/content/drive/MyDrive/Emotion-Recognition/models/emotion_model.h5')

print("Model Saved Successfully!")

Model Saved Successfully!


In [ ]:
#Test Prediction
sample_audio = file_paths[100]

features = extract_mfcc(sample_audio)

features = np.expand_dims(features, axis=0)
features = np.expand_dims(features, axis=2)

prediction = model.predict(features)

predicted_label = np.argmax(prediction)

predicted_emotion = label_encoder.inverse_transform([predicted_label])

print("Predicted Emotion:", predicted_emotion[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
Predicted Emotion: Fearful


In [2]:
!pip install streamlit

In [3]:
!pip install streamlit pyngrok

In [4]:
!npm install -g localtunnel

!pip install streamlit

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 3s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦Requirement already satisfied: streamlit in /usr/local/lib/python3.12/dist-packages (1.57.0)


In [5]:
%%writefile app.py

import streamlit as st
import numpy as np
import librosa
import tempfile
from tensorflow.keras.models import load_model

st.set_page_config(
    page_title="Speech Emotion Recognition",
    page_icon="🎙️",
    layout="centered"
)

# ---------------- CSS ----------------

page_bg = """
<style>

.stApp {
    background: linear-gradient(to right, #141e30, #243b55);
    color: white;
}

.title {
    text-align: center;
    font-size: 55px;
    font-weight: bold;
    margin-top: 20px;
    color: white;
}

.subtitle {
    text-align: center;
    font-size: 20px;
    color: #dddddd;
    margin-bottom: 40px;
}

.result-box {
    background-color: rgba(255,255,255,0.1);
    padding: 25px;
    border-radius: 15px;
    text-align: center;
    margin-top: 30px;
}

.result {
    font-size: 35px;
    font-weight: bold;
    color: #00ffd5;
}

</style>
"""

st.markdown(page_bg, unsafe_allow_html=True)

# ---------------- MODEL ----------------

model = load_model(
    "/content/drive/MyDrive/Emotion-Recognition/models/emotion_model.h5"
)

# ---------------- EMOTIONS ----------------

emotions = [
    "Angry",
    "Calm",
    "Disgust",
    "Fearful",
    "Happy",
    "Neutral",
    "Sad",
    "Surprised"
]

emoji_dict = {
    "Angry": "😠",
    "Calm": "😌",
    "Disgust": "🤢",
    "Fearful": "😨",
    "Happy": "😄",
    "Neutral": "😐",
    "Sad": "😢",
    "Surprised": "😲"
}

# ---------------- FEATURE EXTRACTION ----------------

def extract_features(file_path):

    audio, sample_rate = librosa.load(file_path, sr=None)

    mfccs = librosa.feature.mfcc(
        y=audio,
        sr=sample_rate,
        n_mfcc=40
    )

    mfccs_scaled = np.mean(mfccs.T, axis=0)

    return mfccs_scaled

# ---------------- UI ----------------

st.markdown(
    '<div class="title">🎙️ Speech Emotion Recognition</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">Upload WAV Audio and Detect Human Emotion using AI</div>',
    unsafe_allow_html=True
)

uploaded_file = st.file_uploader(
    "Upload WAV File",
    type=["wav"]
)

# ---------------- PREDICTION ----------------

if uploaded_file is not None:

    st.audio(uploaded_file)

    with tempfile.NamedTemporaryFile(delete=False) as tmp_file:

        tmp_file.write(uploaded_file.read())

        temp_path = tmp_file.name

    features = extract_features(temp_path)

    features = np.expand_dims(features, axis=0)
    features = np.expand_dims(features, axis=2)

    prediction = model.predict(features)

    predicted_index = np.argmax(prediction)

    emotion = emotions[predicted_index]

    emoji = emoji_dict[emotion]

    st.markdown(
        f'''
        <div class="result-box">
            <h2>Detected Emotion</h2>
            <div class="result">{emoji} {emotion}</div>
        </div>
        ''',
        unsafe_allow_html=True
    )

# ---------------- SIDEBAR ----------------

st.sidebar.title("📌 About Project")

st.sidebar.info(
    """
Speech Emotion Recognition using:

✅ MFCC Features
✅ Deep Learning (LSTM)
✅ TensorFlow
✅ Librosa
✅ Streamlit
"""
)

Overwriting app.py


In [6]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://clear-results-add.loca.lt


In [10]:
!rm app.py

In [15]:
!pkill -f streamlit